# Big Data Analytics HW1

## 1. Data Description

**Dataset**: NYC yellow taxi trip data for 2009 Q1

**Files**
- `yellow_tripdata_2009-01.csv`
- `yellow_tripdata_2009-02.csv`
- `yellow_tripdata_2009-03.csv`

**Scale**
- Total size: **7.56 GB**
- Total observations: **41,859,906 rows**

**Main fields**
- `vendor_name`: taxi vendor
- `Trip_Pickup_DateTime`: pickup timestamp
- `Trip_Dropoff_DateTime`: dropoff timestamp
- `Passenger_Count`: number of passengers
- `Trip_Distance`: trip distance in miles
- `Start_Lon`, `Start_Lat`: pickup longitude / latitude
- `End_Lon`, `End_Lat`: dropoff longitude / latitude
- `Payment_Type`: payment method
- `Fare_Amt`: fare amount
- `Tip_Amt`: tip amount
- `Tolls_Amt`: toll amount
- `Total_Amt`: total trip amount

## 2. Project Objectives

This notebook answers three practical questions:

1. **Where are the taxi hotspots?**  
   找出最頻繁上下車的熱點
2. **When are the peak and off-peak hours?**  
   找出離峰和尖峰的時段

3. **How do small-fare and big-fare trips differ?**  
   找出small-fare and big-fare 差異



In [ ]:
#config and lib

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from pyspark.sql import functions as F






#create spark session
spark = (
    SparkSession.builder
    .appName("NYC_Taxi_BigData_HW1_Refactor")
    .config("spark.sql.shuffle.partitions", 200)   #shuffle 200
    .getOrCreate()
)

#define schema

schema = StructType([
    StructField("vendor_name", StringType(), True),
    StructField("Trip_Pickup_DateTime", StringType(), True),
    StructField("Trip_Dropoff_DateTime", StringType(), True),
    StructField("Passenger_Count", IntegerType(), True),
    StructField("Trip_Distance", DoubleType(), True),
    StructField("Start_Lon", DoubleType(), True),
    StructField("Start_Lat", DoubleType(), True),
    StructField("Rate_Code", StringType(), True),
    StructField("store_and_forward", IntegerType(), True),
    StructField("End_Lon", DoubleType(), True),
    StructField("End_Lat", DoubleType(), True),
    StructField("Payment_Type", StringType(), True),
    StructField("Fare_Amt", DoubleType(), True),
    StructField("surcharge", DoubleType(), True),
    StructField("mta_tax", StringType(), True),
    StructField("Tip_Amt", DoubleType(), True),
    StructField("Tolls_Amt", DoubleType(), True),
    StructField("Total_Amt", DoubleType(), True),
])


# Load and Merge the Three Monthly data

In [ ]:
path1 = "gs://bigdata-datalake/raw/yellow_tripdata_2009-01.csv"
path2 = "gs://bigdata-datalake/raw/yellow_tripdata_2009-02.csv"
path3 = "gs://bigdata-datalake/raw/yellow_tripdata_2009-03.csv"
#load data
data1 = spark.read.csv(path1, header=True, schema=schema)
data2 = spark.read.csv(path2, header=True, schema=schema)
data3 = spark.read.csv(path3, header=True, schema=schema)

df = data1.unionByName(data2).unionByName(data3) #merge by name

print("Total rows:", df.count())
df.printSchema()

## Quick Data Audit

df裡發現許多欄位有空值

In [ ]:

audit_df = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])

audit_df.show(truncate=False)

## Data Cleaning

對空值和不合理值進行清洗。

### Rules
- Coordinates not null
- Trip distance 必大於0
- Total amount not null
- Pickup/dropoff timestamps not null

### Geographic range for NYC
範圍必在紐約合理經緯度
- longitude: **-75 to -73**
- latitude: **40.0 to 41.0**

In [ ]:
clean_df = (
    df
    .filter(F.col("Trip_Pickup_DateTime").isNotNull())
    .filter(F.col("Trip_Dropoff_DateTime").isNotNull())
    .filter(F.col("Trip_Distance").isNotNull() & (F.col("Trip_Distance") > 0))
    .filter(F.col("Total_Amt").isNotNull())
    .filter(F.col("Start_Lon").isNotNull() & F.col("Start_Lat").isNotNull())
    .filter(F.col("End_Lon").isNotNull() & F.col("End_Lat").isNotNull())
    .filter((F.col("Start_Lon") > -75) & (F.col("Start_Lon") < -73))
    .filter((F.col("Start_Lat") > 40.0) & (F.col("Start_Lat") < 41.0))
    .filter((F.col("End_Lon") > -75) & (F.col("End_Lon") < -73))
    .filter((F.col("End_Lat") > 40.0) & (F.col("End_Lat") < 41.0))
)

print("Rows after cleaning:", clean_df.count())
clean_df.show(5, truncate=False)


### grid

根據上下車經緯度，切分grid。來定義大致上的方位，以找出hotspot。


In [ ]:
grid_size = 0.01

hotspot_df = (
    clean_df
    .withColumn("pickup_lon_grid", F.round(F.col("Start_Lon") / grid_size) * grid_size)
    .withColumn("pickup_lat_grid", F.round(F.col("Start_Lat") / grid_size) * grid_size)
    .withColumn("dropoff_lon_grid", F.round(F.col("End_Lon") / grid_size) * grid_size)
    .withColumn("dropoff_lat_grid", F.round(F.col("End_Lat") / grid_size) * grid_size)
)

### Top Pickup Hotspots & Top Dropoff Hotspots

In [ ]:
pickup_hotspots = (
    hotspot_df
    .groupBy("pickup_lon_grid", "pickup_lat_grid")
    .count()
    .orderBy(F.desc("count"))
)

pickup_hotspots.show(10, truncate=False)



In [ ]:
dropoff_hotspots = (
    hotspot_df
    .groupBy("dropoff_lon_grid", "dropoff_lat_grid")
    .count()
    .orderBy(F.desc("count"))
)

dropoff_hotspots.show(10, truncate=False)

## Peak vs Off-Peak Analysis


In [ ]:
time_df = (
    clean_df
    .withColumn("pickup_ts", F.to_timestamp("Trip_Pickup_DateTime")) #轉timestamp
    .withColumn("pickup_hour", F.hour("pickup_ts")) #以小時為單位
)

hourly_demand = (
    time_df
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

hourly_demand.show(24)

### Peak and Off-Peak Hours

In [ ]:
peak_hour = hourly_demand.orderBy(F.desc("count")).limit(1)
offpeak_hour = hourly_demand.orderBy(F.asc("count")).limit(1)

print("Peak hour:") #19:00
peak_hour.show()

print("Off-peak hour:") #5:00
offpeak_hour.show()

## Big Amount vs Small Amount

- Small fare: `Total_Amt < mean - std`
- Big fare: `Total_Amt > mean + std`

In [ ]:
fare_stats = clean_df.select(
    F.avg("Total_Amt").alias("mean_amt"),
    F.stddev("Total_Amt").alias("std_amt")
)

fare_stats.show()

In [ ]:
stats = fare_stats.collect()[0]
mean_amt = stats["mean_amt"]
std_amt = stats["std_amt"]

small_threshold = mean_amt - std_amt
big_threshold = mean_amt + std_amt

print("mean_amt =", mean_amt) #10.49
print("std_amt  =", std_amt) #8.39
print("small threshold =", small_threshold)
print("big threshold   =", big_threshold)

In [ ]:
#seperate df to small fare and big fare
small_df = clean_df.filter(F.col("Total_Amt") < small_threshold)
big_df = clean_df.filter(F.col("Total_Amt") > big_threshold)

print("Small-fare trips:", small_df.count())
print("Big-fare trips:", big_df.count())

### Compare by Pickup Hour
- groupby
- count

In [ ]:
small_by_hour = (
    small_df
    .withColumn("pickup_ts", F.to_timestamp("Trip_Pickup_DateTime"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

big_by_hour = (
    big_df
    .withColumn("pickup_ts", F.to_timestamp("Trip_Pickup_DateTime"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

print("Small-fare trips by hour")
small_by_hour.show(24)

print("Big-fare trips by hour")
big_by_hour.show(24)

### Compare Average Trip Distance

In [ ]:
small_df.select(F.avg("Trip_Distance").alias("avg_small_trip_distance")).show() #5mile
big_df.select(F.avg("Trip_Distance").alias("avg_big_trip_distance")).show() #0.53 mile